#Extract Data and Load in Bronze Layer

In [0]:
medical_insurance.default.bed

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp, lit, to_date, col
from datetime import datetime
import traceback
import uuid

# Generate unique pipeline run ID for this execution
pipeline_run_id = str(uuid.uuid4())
print(f"Pipeline Run ID: {pipeline_run_id}\n")

# Source path
source_path = "/Volumes/workspace/bronze/source_systems/Raw Data/"

# Get all CSV files
files = dbutils.fs.ls(source_path)
csv_files = [f.name for f in files if f.name.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files to process\n")

# Track results for quality summary
results = []
quality_metrics = []

for file_name in csv_files:
    table_name = file_name.replace('.csv', '').lower()
    file_path = source_path + file_name
    
    try:
        start_time = datetime.now()
        
        # Get file metadata
        file_info = dbutils.fs.ls(file_path)[0]
        file_modified_time = datetime.fromtimestamp(file_info.modificationTime / 1000)
        
        # Read CSV
        df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
        
        # Add audit/metadata columns
        df_with_audit = (df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("ingestion_date", to_date(current_timestamp()))  # For partitioning
            .withColumn("source_file", lit(file_name))
            .withColumn("source_file_path", lit(file_path))
            .withColumn("source_file_modified_time", lit(file_modified_time))
            .withColumn("pipeline_run_id", lit(pipeline_run_id))
        )
        
        # Get row count and data quality metrics
        row_count = df_with_audit.count()
        col_count = len(df.columns)  # Original columns without audit
        
        # Calculate null counts for quality tracking
        null_counts = {}
        for column in df.columns:
            null_count = df.filter(col(column).isNull()).count()
            null_counts[column] = null_count
        
        total_nulls = sum(null_counts.values())
        
        # Write to bronze layer with partitioning
        df_with_audit.write.mode('overwrite').partitionBy("ingestion_date").saveAsTable(f'bronze.{table_name}')
        
        # Set Delta Lake properties for schema evolution and optimization
        spark.sql(f"""
            ALTER TABLE bronze.{table_name} SET TBLPROPERTIES (
                'delta.enableChangeDataFeed' = 'true',
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true'
            )
        """)
        
        # Add table comment for data lineage
        spark.sql(f"""
            COMMENT ON TABLE bronze.{table_name} IS 
            'Bronze layer table loaded from {file_path}. Contains raw data with audit columns for tracking. Partitioned by ingestion_date.'
        """)
        
        # Add Unity Catalog tags for lineage and categorization
        spark.sql(f"""
            ALTER TABLE bronze.{table_name} SET TAGS (
                'layer' = 'bronze',
                'domain' = 'healthcare',
                'source_system' = 'csv_files',
                'data_classification' = 'raw'
            )
        """)
        
        # Validate by reading back
        validation_df = spark.table(f'bronze.{table_name}')
        validation_count = validation_df.count()
        
        # Check if counts match
        status = "✓ SUCCESS" if row_count == validation_count else "✗ MISMATCH"
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        # Store results
        results.append({
            'file': file_name,
            'table': f'bronze.{table_name}',
            'status': status,
            'rows': row_count,
            'columns': col_count,
            'validated_rows': validation_count,
            'duration_sec': round(duration, 2)
        })
        
        # Store quality metrics
        quality_metrics.append({
            'pipeline_run_id': pipeline_run_id,
            'table_name': f'bronze.{table_name}',
            'load_timestamp': datetime.now(),
            'row_count': row_count,
            'column_count': col_count,
            'total_null_values': total_nulls,
            'null_percentage': round((total_nulls / (row_count * col_count) * 100) if row_count > 0 else 0, 2),
            'file_size_bytes': file_info.size,
            'processing_duration_sec': round(duration, 2),
            'status': status
        })
        
        print(f"{status} | {table_name:30} | Rows: {row_count:>10,} | Cols: {col_count:>3} | Nulls: {total_nulls:>8,} | {duration:.2f}s")
        
    except Exception as e:
        results.append({
            'file': file_name,
            'table': f'bronze.{table_name}',
            'status': '✗ FAILED',
            'rows': 0,
            'columns': 0,
            'validated_rows': 0,
            'duration_sec': 0,
            'error': str(e)
        })
        
        quality_metrics.append({
            'pipeline_run_id': pipeline_run_id,
            'table_name': f'bronze.{table_name}',
            'load_timestamp': datetime.now(),
            'row_count': 0,
            'column_count': 0,
            'total_null_values': 0,
            'null_percentage': 0,
            'file_size_bytes': 0,
            'processing_duration_sec': 0,
            'status': '✗ FAILED',
            'error_message': str(e)
        })
        
        print(f"✗ FAILED | {table_name:30} | Error: {str(e)}")

# Create and save data quality summary table
quality_df = spark.createDataFrame(quality_metrics)
quality_df.write.mode('append').saveAsTable('bronze.data_quality_metrics')

print("\n" + "="*80)
print("Data quality metrics saved to: bronze.data_quality_metrics")
print("="*80)

# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

success_count = len([r for r in results if r['status'] == '✓ SUCCESS'])
failed_count = len([r for r in results if r['status'] == '✗ FAILED'])
total_rows = sum([r['rows'] for r in results])

print(f"Total files processed: {len(results)}")
print(f"Successful loads: {success_count}")
print(f"Failed loads: {failed_count}")
print(f"Total rows loaded: {total_rows:,}")
print(f"\nBronze Layer Features Applied:")
print(f"  ✓ Audit columns: ingestion_timestamp, source_file, pipeline_run_id")
print(f"  ✓ Partitioning: By ingestion_date")
print(f"  ✓ Delta properties: Schema evolution, CDC, Auto-optimize")
print(f"  ✓ Unity Catalog tags: layer, domain, source_system")
print(f"  ✓ Data lineage: Table comments")
print(f"  ✓ Quality tracking: Null counts, validation checks")

if failed_count > 0:
    print("\nFailed tables:")
    for r in results:
        if r['status'] == '✗ FAILED':
            print(f"  - {r['table']}: {r.get('error', 'Unknown error')}")

# Create results DataFrame for easy viewing
results_df = spark.createDataFrame(results)
display(results_df)

In [0]:
# Query the data quality metrics table to see tracking history
quality_summary = spark.sql("""
    SELECT 
        pipeline_run_id,
        table_name,
        load_timestamp,
        row_count,
        column_count,
        total_null_values,
        null_percentage,
        ROUND(file_size_bytes / 1024 / 1024, 2) as file_size_mb,
        processing_duration_sec,
        status
    FROM bronze.data_quality_metrics
    ORDER BY load_timestamp DESC, table_name
""")

display(quality_summary)

In [0]:
# Verify Delta properties and UC tags for a sample table
sample_table = 'bronze.patients'

print("=" * 80)
print(f"Configuration for: {sample_table}")
print("=" * 80)

# Show table properties
print("\n1. DELTA LAKE PROPERTIES:")
properties = spark.sql(f"SHOW TBLPROPERTIES {sample_table}").collect()
for prop in properties:
    if 'delta' in prop.key.lower() or 'autoOptimize' in prop.key:
        print(f"   {prop.key} = {prop.value}")

# Show table tags
print("\n2. UNITY CATALOG TAGS:")
tags = spark.sql(f"SHOW TAGS ON TABLE {sample_table}").collect()
for tag in tags:
    print(f"   {tag.tag_name} = {tag.tag_value}")

# Show table comment
print("\n3. TABLE COMMENT (Data Lineage):")
table_info = spark.sql(f"DESCRIBE TABLE EXTENDED {sample_table}").filter(col("col_name") == "Comment").collect()
if table_info:
    print(f"   {table_info[0].data_type}")

# Show partition info
print("\n4. PARTITIONING:")
partition_info = spark.sql(f"DESCRIBE TABLE EXTENDED {sample_table}").filter(col("col_name") == "# Partition Information").collect()
if partition_info:
    print("   Partitioned by: ingestion_date")
else:
    print("   Not partitioned")

# Show audit columns
print("\n5. AUDIT COLUMNS:")
audit_columns = spark.table(sample_table).columns
audit_cols = [c for c in audit_columns if c in ['ingestion_timestamp', 'ingestion_date', 'source_file', 'source_file_path', 'source_file_modified_time', 'pipeline_run_id']]
for col_name in audit_cols:
    print(f"   ✓ {col_name}")

print("\n" + "=" * 80)